# Test simulation for Valentine paper recreation
## Imports

In [40]:
import cmath
import math

import matplotlib.pyplot as plt
import numpy as np

import meep as mp
from meep import materials


## Setup 
### Cell dimensions

In [41]:
resolution = 60  # pixels/μm

dpml = 2.0  # PML thickness
dsub = 3.0  # substrate thickness
dencap = 3.0 # encapsulant total thickness
ppx = 0.60  # pillar period in x
ppy = 0.600 # pillar period in y
ph = 0.440  # pillar height
pr = 0.140 # pillar radius



sz = dpml + dsub + dencap + dpml
sy = ppy
sx = ppx

cell_size = mp.Vector3(sx, sy, sz)
pml_layers = [mp.PML(thickness=dpml, direction=mp.Z)]

### Source control 

In [43]:
wvl_min = 0.5  # min wavelength
wvl_max = 1  # max wavelength

# wvl = 1.1

fmin = 1 / wvl_max  # min frequency
fmax = 1 / wvl_min  # max frequency
fcen = 0.5 * (fmin + fmax)  # center frequency

# fcen = 1 / wvl

df = fmax - fmin  # frequency width

# df = 0.05 * fcen

src_pt = mp.Vector3(0, 0, 0.5 * sz - dpml - 0.5 * dencap)
sources = [
    mp.Source(
        mp.GaussianSource(fcen, fwidth=df),
        component=mp.Ex,
        center=src_pt,
        size=mp.Vector3(sx, sy, 0),
    )
]

### Cell setup

In [46]:
k_point = mp.Vector3(0, 0, 0)

symmetries = [mp.Mirror(mp.X), mp.Mirror(mp.Y), mp.Rotate4(mp.Z)]

glass = materials.fused_quartz
pmma = materials.PMMA
silicon = materials.cSi

geometry = [
    mp.Block(
        material=glass,
        size=mp.Vector3(mp.inf, mp.inf, dpml + dsub),
        center=mp.Vector3(0, 0, -0.5 * sz + dpml + 0.5 * dsub),
    ),
    mp.Block(
        material=pmma,
        size=mp.Vector3(mp.inf, mp.inf, dpml + dencap),
        center=mp.Vector3(0, 0, 0.5 * sz - dpml - 0.5 * dencap),
    ),
    mp.Cylinder(
        material = silicon,
        radius = pr, 
        height = ph, 
        axis = mp.Vector3(0, 0, 1),
        center = mp.Vector3(0, 0, 0.5 * ph)
    )
]


### Simulation setup

In [48]:
sim = mp.Simulation(
    resolution=resolution,
    cell_size=cell_size,
    boundary_layers=pml_layers,
    geometry=geometry,
    k_point=k_point,
    sources=sources,
    symmetries=symmetries,
)

nfreq = 1
mon_pt = mp.Vector3(0, 0, -0.5 * sz + dpml + 0.5 * dsub)
flux_mon = sim.add_flux(
    fcen, df, nfreq, mp.FluxRegion(center=mon_pt, size=mp.Vector3(sx, sy, 0))
)

mode_mon = sim.add_flux(
    fcen, df, nfreq, mp.FluxRegion(center=mon_pt, size=mp.Vector3(sx, sy, 0))
)


## running the code

In [49]:
sim.plot3D()

RFBOutputContext()

     block, center = (0,0,-1.5)
          size (1e+20,1e+20,5)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (1,1,1)
     block, center = (0,0,1.5)
          size (1e+20,1e+20,5)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (1,1,1)
     cylinder, center = (0,0,0.22)
          radius 0.14, height 0.44, axis (0, 0, 1)
          dielectric constant epsilon diagonal = (1,1,1)


In [51]:
sim.plot_fields()

In [50]:
sim.run(until_after_sources=mp.stop_when_fields_decayed(25, mp.Ex, mon_pt, 1e-5))

-----------
Initializing structure...
Halving computational cell along direction x
Halving computational cell along direction y
time for choose_chunkdivision = 0.000794888 s
Working in 3D dimensions.
Computational cell is 0.6 x 0.6 x 10 with resolution 60
     block, center = (0,0,-1.5)
          size (1e+20,1e+20,5)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (1,1,1)
     block, center = (0,0,1.5)
          size (1e+20,1e+20,5)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (1,1,1)
     cylinder, center = (0,0,0.22)
          radius 0.14, height 0.44, axis (0, 0, 1)
          dielectric constant epsilon diagonal = (1,1,1)
time for set_epsilon = 0.267276 s
lorentzian susceptibility: frequency=1.73, gamma=5
lorentzian susceptibility: frequency=2.76, gamma=0.126
lorentzian susceptibility: frequency=3.64, gamma=0
lorentzian susceptibility: frequency=9.4018, gamma=0
lorentzian susceptibility: frequency

KeyboardInterrupt: 